# Chương 3 — Mạng nơ-ron tích chập: im2col/col2im và ba cách cài đặt trên RetinaMNIST

In [1]:
# CNN-00 · Thiết lập đường dẫn: mã chương dùng chung gói của tiểu luận (src/dm) và mã kế thừa của nhóm (src/tl01, tl03)
import os, sys, json, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
warnings.filterwarnings("ignore")
TLF = Path.cwd().parent
for p in ["src", "src/tl01", "src/tl03"]:
    sys.path.insert(0, str(TLF / p))
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
print("Thư mục tiểu luận:", TLF)

Thư mục tiểu luận: /home/ngtukien/Projects/MultiAgent/tieuluan_final


Notebook kiểm chứng tích chập viết tay bằng `im2col`/`col2im`, đối chiếu đạo hàm giải tích với sai phân hữu hạn, và so sánh 5 mô hình CNN (NumPy, Keras, PyTorch) trên bộ ảnh đáy mắt **RetinaMNIST** cùng hai bộ ảnh đối chiếu **MNIST**, **Fashion-MNIST**.

In [2]:
# CNN-01 · Nạp dữ liệu ảnh đáy mắt RetinaMNIST và xem phân bố 5 mức bệnh võng mạc ĐTĐ
from tl3.ch3_data import load_raw, CLASSES
Xtr, ytr, Xte, yte = load_raw("retina")
print("Train+Val:", Xtr.shape, "Test:", Xte.shape)
dist = pd.Series(yte).value_counts().sort_index()
for k, cnt in dist.items():
    print(f"  Mức {k} ({CLASSES['retina'][k]}): {cnt} ảnh ({cnt/len(yte):.1%})")

Train+Val: (1200, 128, 128, 3) Test: (400, 128, 128, 3)
  Mức 0 (Mức 0 — không bệnh): 174 ảnh (43.5%)
  Mức 1 (Mức 1 — nhẹ): 46 ảnh (11.5%)
  Mức 2 (Mức 2 — trung bình): 92 ảnh (23.0%)
  Mức 3 (Mức 3 — nặng): 68 ảnh (17.0%)
  Mức 4 (Mức 4 — tăng sinh): 20 ảnh (5.0%)


In [3]:
# CNN-02 · Kiểm tra đạo hàm giải tích của conv_backward (im2col/col2im) so với sai phân hữu hạn
from tl3.ch3_scratch import conv_forward, conv_backward
rng = np.random.default_rng(42)
X = rng.standard_normal((2, 3, 6, 6))
W = rng.standard_normal((4, 3, 3, 3)) * 0.1
b = rng.standard_normal(4) * 0.1
out, cache = conv_forward(X, W, b, pad=1)
dOut = rng.standard_normal(out.shape)
dX, dW, db = conv_backward(dOut, cache)
eps = 1e-5
num_dW = np.zeros_like(W)
for idx in np.ndindex(W.shape):
    W[idx] += eps; p1, _ = conv_forward(X, W, b, pad=1)
    W[idx] -= 2*eps; p2, _ = conv_forward(X, W, b, pad=1)
    W[idx] += eps
    num_dW[idx] = np.sum((p1 - p2) * dOut) / (2 * eps)
rel_err = np.max(np.abs(dW - num_dW) / np.maximum(1e-8, np.abs(dW) + np.abs(num_dW)))
print(f"Kích thước đầu ra conv: {out.shape}, sai số tương đối cực đại của dW = {rel_err:.2e}")

Kích thước đầu ra conv: (2, 4, 6, 6), sai số tương đối cực đại của dW = 8.28e-11


In [4]:
# CNN-03 · Đánh giá 5 mô hình đã huấn luyện trên 400 ảnh kiểm tra RetinaMNIST (Accuracy, Macro-F1, QWK)
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
from tl3.serve_ch3 import _load, preprocess
from PIL import Image
rows = []
for mid in ["scratch", "tf_basic", "torch_basic", "tf_improved", "torch_improved"]:
    fn, sz = _load("retina", mid)
    xs = np.concatenate([preprocess("retina", Image.fromarray(im), sz) for im in Xte], axis=0)
    pred = fn(xs).argmax(axis=1)
    rows.append({"mô hình": mid, "kích thước": f"{sz}x{sz}",
                 "acc": accuracy_score(yte, pred),
                 "macro_f1": f1_score(yte, pred, average="macro"),
                 "qwk": cohen_kappa_score(yte, pred, weights="quadratic")})
pd.DataFrame(rows).round(4)

I0000 00:00:1793841938.634670  329815 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1793841939.690204  329815 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


,mô hình,kích thước,acc,macro_f1,qwk
0,scratch,48x48,0.5375,0.3467,0.5216
1,tf_basic,48x48,0.4875,0.3497,0.4551
2,torch_basic,48x48,0.5550,0.3786,0.5892
3,tf_improved,96x96,0.5200,0.2471,0.5172
4,torch_improved,96x96,0.5500,0.3268,0.5002


In [5]:
# CNN-04 · Đối chiếu cấu trúc hàm hợp 5 khối của BasicCNN và số tham số từng lớp
from tl3.ch3_torch import BasicCNN
net = BasicCNN(in_ch=3, n_classes=5, size=48)
total = sum(p.numel() for p in net.parameters())
for name, mod in net.named_children():
    n_p = sum(p.numel() for p in mod.parameters())
    print(f"Lớp {name:4s} ({mod.__class__.__name__:10s}): {n_p:9,d} tham số ({n_p/total:6.2%})")
print(f"Tổng cộng: {total:,d} tham số")

Lớp features (Sequential):    19,392 tham số ( 1.62%)
Lớp classifier (Sequential): 1,180,421 tham số (98.38%)
Tổng cộng: 1,199,813 tham số
